# 📖 00 · Project Overview

> **Chapter 0 of the project narrative.** This notebook is the map.
> Every other notebook is a chapter; every section below links to
> one.

**Goal.** Classify human missense variants as pathogenic or benign
with honest evaluation — paralog-aware splits, zero ClinVar
contamination, 1,000-bootstrap CIs, published-tool comparison on our
exact test set.

**Current state.** Phase 1 complete, Phase 2 step 1 (gnomAD
constraint) merged, Phase 2 step 2 (ESM-2 zero-shot) proof-of-concept
done. Stage 1 (baselines), Stage 2.4 (SHAP), Stage 3 (calibration)
landed. Stage 2.1 (full-training-set ESM-2) running on Colab.

---

## Chapter index

| # | Notebook | What it shows |
|---|---|---|
| 01 | Data sources & cleaning | ClinVar / gnomAD / dbNSFP — how 195K missense variants were assembled. |
| 02 | Feature engineering | What 39 features ended up in the model and how each one is computed. |
| 03 | **The leakage hunt** | 0.955 → 0.838 — three distinct leakage sources, each caught, each measured. |
| 04 | Model training & selection | Optuna TPE, 40 trials, PR-AUC objective, threshold sweep. |
| 05 | Evaluation & calibration | Bootstrap CIs, Brier decomposition, reliability triptych, operating points. |
| 06 | SHAP & interpretability | TreeSHAP on 2,000 test rows; top-20 features; confident errors. |
| 07 | External validation — denovo-db | 0.47 → 0.57 ROC on unseen families after constraint features. |
| 08 | Baseline comparison | SIFT / PolyPhen-2 / AlphaMissense on our exact test. |
| 09 | ESM-2 zero-shot proof-of-concept | Rank-fusion with XGBoost; decision to scale to full training set. |

---

## Design contract

Every notebook in this series:

1. **Loads from `results/metrics/*.csv` or `data/splits/*.parquet`** —
   no heavy re-computation inside the notebook. This keeps execution
   under ~30 s end-to-end.
2. **Cites the committed CHANGELOG entry** that produced the artifacts
   it displays.
3. **Ends with a "reproduce" command** — the script in `src/` or
   `scripts/` that regenerates the CSVs/figures from scratch.
4. **Uses `seed=42` everywhere.** Outputs are byte-for-byte
   reproducible if you reinstall from `requirements-lock.txt` and
   rerun the upstream scripts.


In [ ]:
# Imports + reproducibility seeds
from __future__ import annotations

import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

REPO = Path.cwd().parent
METRICS = REPO / "results/metrics"
FIGURES = REPO / "results/figures"

print(f"Repo root: {REPO}")
print(f"Artifacts:  {METRICS.relative_to(REPO)} + {FIGURES.relative_to(REPO)}")

## Headline numbers (auto-loaded from committed CSVs)

The cells below read from the same CSVs the main README renders,
so these numbers cannot drift out of sync with the documentation.

In [ ]:
split_metrics = pd.read_csv(METRICS / "xgboost_split_metrics.csv")
test = split_metrics[split_metrics["split"] == "test"].iloc[0]
val = split_metrics[split_metrics["split"] == "val"].iloc[0]

print("ClinVar test (paralog-disjoint)")
print(f"  ROC-AUC  {test['roc_auc']:.4f}")
print(f"  PR-AUC   {test['pr_auc']:.4f}")
print(f"  F1       {test['f1']:.4f}")
print(f"  Brier    {test['brier_loss']:.4f}")
print(f"  Support  {int(test['support']):,} variants")

In [ ]:
# External denovo-db — the external-generalization number
try:
    ext = pd.read_csv(METRICS / "external_denovo_db_metrics.csv")
    print("denovo-db external validation")
    for _, r in ext.iterrows():
        print(f"  {r['slice']:>25s}  n={int(r['n']):>4d}  "
              f"ROC {r['roc_auc']:.3f} [{r['roc_auc_ci_lo']:.3f}, {r['roc_auc_ci_hi']:.3f}]  "
              f"PR {r['pr_auc']:.3f} [{r['pr_auc_ci_lo']:.3f}, {r['pr_auc_ci_hi']:.3f}]")
except FileNotFoundError:
    print("denovo-db metrics not present — run scripts/evaluate_external.py")

In [ ]:
# Baselines on the same test split
try:
    bl = pd.read_csv(METRICS / "baselines_comparison.csv")
    bl = bl[bl["slice"] == "clinvar_test"].copy()
    bl["display"] = bl.apply(
        lambda r: f"ROC {r['roc_auc']:.3f}  PR {r['pr_auc']:.3f}  cov {r['coverage']:.0%}",
        axis=1,
    )
    print("Baselines on ClinVar test")
    for _, r in bl.iterrows():
        print(f"  {r['baseline_display_name']:<16s}  {r['display']}")
    print(f"  {'XGBoost (ours)':<16s}  ROC {test['roc_auc']:.3f}  PR {test['pr_auc']:.3f}  cov 100%")
except FileNotFoundError:
    print("baselines comparison not present — run scripts/run_baselines.py")

### Interpretation

The first number reviewers ask about is the ClinVar test score,
typically with the implicit assumption it's comparable to published
benchmarks. It isn't — most published benchmarks lack paralog-aware
guards. The honest comparison comes from the baseline table above,
where every row was scored on *our exact test*.

The external number is what matters clinically: how does a
ClinVar-trained classifier generalize to de-novo variants in unseen
gene families? The answer is "at chance before gnomAD constraint, above
chance after" — the whole Phase 2 roadmap is motivated by that gap.

---

## Reproduce this notebook

```bash
# Regenerate all upstream CSVs:
python -m src.training              # xgboost_split_metrics.csv
python -m src.evaluate_baseline     # calibration + operating points
python scripts/evaluate_external.py --only denovo_db --use-vep
python scripts/run_baselines.py     # baselines_comparison.csv
```

Then re-run this notebook top to bottom.